# High-Pay Occupation Prediction with Logistic Regression

## Project Overview

This project uses **logistic regression** to predict whether an occupation is classified as **high-paying** based on employment, job concentration, state, and occupation-group characteristics.

The analysis includes data cleaning, exploratory analysis, feature engineering, model diagnostics, nonlinear-effect testing, model evaluation, odds-ratio interpretation, and state/occupation-group predictions.

> **Model:** Logistic regression  
> **Target:** `High_Pay`  
> **Classification threshold:** 0.50  
> **Train/test split:** 80/20 stratified split  
> **Random seed:** 123


## Research Question

**Can employment characteristics, occupation group, and state help predict whether an occupation is above the overall median annual wage?**

The binary target `High_Pay` is created from the median `A_MEDIAN` value in the cleaned dataset.


## 1. Setup and Packages

In [ ]:
# ============================================================
# Setup: packages
# ============================================================
library(tidyverse)
library(scales)
library(readxl)
library(dplyr)
library(caret)
library(tidymodels)
library(pROC)
library(car)
library(broom)
library(GGally)
library(purrr)
library(corrplot)
library(ggplot2)
library(knitr)
library(gt)


## 2. Dataset

The analysis uses the `state_M25.xlsx` dataset. For GitHub, the notebook expects the workbook to be placed in the project's `data/` directory.

The analysis focuses on **detailed occupations** rather than aggregate occupation rows.


In [ ]:
# Dataset path
# Place the downloaded state_M25.xlsx file in the project's data/ folder.
file_path <- "data/state_M25.xlsx"

if (!file.exists(file_path)) {
  stop(
    "Dataset not found. Place state_M25.xlsx in the data/ folder. ",
    "See the project README/data instructions for the source."
  )
}

sheet_names <- excel_sheets(file_path)
print(sheet_names)

df <- read_excel(file_path)

glimpse(df)
head(df)


## 3. Data Preparation

### 3.1 Select Detailed Occupations and Core Variables

In [ ]:
# 1. Select chosen columns and clean numeric features
df_selected <- df %>%
  # Filter for detailed occupations to avoid aggregate row overlap
  filter(O_GROUP == "detailed") %>%
  
  # Keep only your chosen columns
  select(
    PRIM_STATE, 
    NAICS, 
    NAICS_TITLE, 
    OCC_CODE, 
    OCC_TITLE, 
    TOT_EMP, 
    JOBS_1000, 
    LOC_QUOTIENT, 
    A_MEDIAN
  ) %>%
  
  # Convert numeric columns (converting text symbols like '*' or '#' to NA)
  mutate(
    A_MEDIAN_num     = suppressWarnings(parse_number(A_MEDIAN)),
    TOT_EMP_num      = suppressWarnings(parse_number(TOT_EMP)),
    JOBS_1000_num    = suppressWarnings(parse_number(JOBS_1000)),
    LOC_QUOTIENT_num = suppressWarnings(parse_number(LOC_QUOTIENT))
  ) %>%
  
  # Remove original character versions of the numeric columns
  select(-A_MEDIAN, -TOT_EMP, -JOBS_1000, -LOC_QUOTIENT) %>%
  
  # Remove rows with NA in key variables
  filter(!is.na(A_MEDIAN_num))

# 2. Inspect the cleaned dataset
glimpse(df_selected)

### 3.2 Missing Values

In [ ]:
# 1. Inspect count of missing values per column
sapply(df_selected, function(x) sum(is.na(x)))

# 2. Drop rows containing missing values in any column
df_clean <- df_selected %>%
  drop_na()

# 3. Confirm missing values are removed and view row count
cat("Remaining complete rows:", nrow(df_selected), "\n")


### 3.3 Duplicate Records

In [ ]:
# 1. Count total duplicated rows
num_duplicates <- sum(duplicated(df_clean))
cat("Number of duplicate rows:", num_duplicates, "\n")

# 2. Remove duplicates if any exist
df_clean <- df_clean %>%
  distinct()

# 3. Confirm dataset dimensions after removing duplicates
cat("Remaining rows after removing duplicates:", nrow(df_clean), "\n")

### 3.4 Data Types and Unique Values

In [ ]:
# Check data types and count of unique values for each column
df_clean %>%
  summarise(across(everything(), list(
    data_type = ~ class(.x)[1],
    unique_count = ~ n_distinct(.x)
  ))) %>%
  pivot_longer(
    cols = everything(),
    names_to = c("column", ".value"),
    names_sep = "_(?=[^_]+$)"
  )

### 3.5 Remove Constant Variables

`NAICS` and `NAICS_TITLE` contain only one unique value in the selected data and therefore provide no predictive information.


In [ ]:
# Remove NAICS columns because they contain only one unique value
df_clean <- df_clean %>%
  select(-NAICS, -NAICS_TITLE)
# Confirm remaining columns
colnames(df_clean)

## 4. Exploratory Data Analysis

### 4.1 Numeric Feature Distributions and Outliers

In [ ]:
# check the numeric variables for outliers and their distributions
summary(df_clean[c("A_MEDIAN_num","TOT_EMP_num","JOBS_1000_num","LOC_QUOTIENT_num")])

### 4.2 Boxplots

In [ ]:
# Reshape data and generate boxplots
df_clean %>%
  select(A_MEDIAN_num, TOT_EMP_num, JOBS_1000_num, LOC_QUOTIENT_num) %>%
  pivot_longer(
    cols = everything(), 
    names_to = "Variable", 
    values_to = "Value"
  ) %>%
  ggplot(aes(x = Variable, y = Value, fill = Variable)) +
  geom_boxplot(alpha = 0.7, show.legend = FALSE, outlier.color = "red", outlier.size = 1) +
  facet_wrap(~ Variable, scales = "free", ncol = 2) +
  theme_minimal() +
  labs(
    title = "Boxplots of Numeric Features (Outlier Detection)",
    x = "",
    y = "Value"
  )

### 4.3 Value Checks

The analysis checks for negative, zero, and positive values in the numeric variables.


In [ ]:
library(purrr)

# Check count of negative, zero, and positive values for each numeric column
df_clean %>%
  select(A_MEDIAN_num, TOT_EMP_num, JOBS_1000_num, LOC_QUOTIENT_num) %>%
  map_df(~ data.frame(
    Negative = sum(.x < 0, na.rm = TRUE),
    Zero     = sum(.x == 0, na.rm = TRUE),
    Positive = sum(.x > 0, na.rm = TRUE)
  ), .id = "Variable")

### 4.4 Rare Categories

Category frequencies are examined for state and occupation variables, including the derived two-digit occupation group.


In [ ]:
# 1. Summary function for category frequency and low-count thresholds
check_rare_categories <- function(df, col_name, threshold_pct = 0.01) {
  df %>%
    group_by(across(all_of(col_name))) %>%
    summarise(Count = n(), .groups = "drop") %>%
    mutate(
      Percent = (Count / sum(Count)) * 100,
      Is_Rare = Percent < (threshold_pct * 100)
    ) %>%
    arrange(Count)
}

# 2. Check PRIM_STATE for rare levels
state_summary <- check_rare_categories(df_clean, "PRIM_STATE")
cat("--- PRIM_STATE Rare Categories (<1% of data) ---\n")
print(filter(state_summary, Is_Rare))
cat("Total States/Territories:", nrow(state_summary), "\n\n")

# 3. Check OCC_CODE for rare detailed occupations
occ_summary <- check_rare_categories(df_clean, "OCC_CODE")
cat("--- OCC_CODE Rare Categories (<0.1% of data) ---\n")
print(filter(occ_summary, Percent < 0.1))
cat("Total Unique OCC_CODEs:", nrow(occ_summary), "\n\n")

# 4. Check 2-digit Broad Occupation Groups (OCC_GROUP)
df_clean <- df_clean %>% mutate(OCC_GROUP = substr(OCC_CODE, 1, 2))
occ_group_summary <- check_rare_categories(df_clean, "OCC_GROUP")
cat("--- OCC_GROUP (2-Digit) Distribution ---\n")
print(occ_group_summary)

### 4.5 Correlation Analysis

In [ ]:
library(corrplot)
# 1. Select numeric variables
numeric_cols <- df_clean %>%
  select(A_MEDIAN_num, TOT_EMP_num, JOBS_1000_num, LOC_QUOTIENT_num)

# 2. Compute Pearson correlation matrix
cor_matrix <- cor(numeric_cols, use = "complete.obs")
print(round(cor_matrix, 3))

# 3. Visualize correlation matrix
corrplot(
  cor_matrix, 
  method = "color", 
  type = "upper", 
  addCoef.col = "black", 
  tl.col = "black", 
  tl.srt = 45,
  title = "Correlation Matrix of Numeric Features",
  mar = c(0, 0, 2, 0)
)

# 4. Fit linear model to compute baseline VIF scores
vif_model <- lm(A_MEDIAN_num ~ TOT_EMP_num + JOBS_1000_num + LOC_QUOTIENT_num, data = df_clean)
vif(vif_model)

## 5. Target Variable: `High_Pay`

The target is created by comparing annual median wages with the overall median wage in the cleaned dataset.

- `High_Pay = 1`: annual median wage is at or above the median threshold
- `High_Pay = 0`: annual median wage is below the median threshold

The reported median wage threshold is **$59,680**, producing an approximately balanced target.


In [ ]:
# 1. Compute the median salary threshold across all valid records
pay_threshold <- median(df_clean$A_MEDIAN_num, na.rm = TRUE)

# 2. Binarize A_MEDIAN_num into the target variable High_Pay
df_clean <- df_clean %>%
  mutate(
    High_Pay = factor(
      if_else(A_MEDIAN_num >= pay_threshold, "High", "Not_High"),
      levels = c("Not_High", "High") # Sets 'Not_High' as baseline level
    )
  )

# 3. Output threshold details and evaluate target class balance
cat("Median Salary Threshold Cutoff: $", format(pay_threshold, big.mark = ","), "\n\n")

# Frequency table and class proportions
table(df_clean$High_Pay)
prop.table(table(df_clean$High_Pay))

## 6. Feature Engineering

### 6.1 Remove Target-Leaking and Non-Predictive Fields

`A_MEDIAN_num` is removed because it directly defines the target. `OCC_TITLE` is also excluded from the model.


In [ ]:
# Remove A_MEDIAN_num and non-predictive title columns from the dataset
df_model_prep <- df_clean %>%
  select(-A_MEDIAN_num, -OCC_TITLE)

# Confirm target and feature set columns
colnames(df_model_prep)

### 6.2 Log Transformations

Right-skewed numeric predictors are transformed with `log1p()`:

- `TOT_EMP_num` → `log_TOT_EMP`
- `JOBS_1000_num` → `log_JOBS_1000`
- `LOC_QUOTIENT_num` → `log_LOC_QUOTIENT`


In [ ]:
# Apply log1p transformation to handle right-skewness
df_model_prep <- df_model_prep %>%
  mutate(
    log_TOT_EMP      = log1p(TOT_EMP_num),
    log_JOBS_1000    = log1p(JOBS_1000_num),
    log_LOC_QUOTIENT = log1p(LOC_QUOTIENT_num)
  ) %>%
  # Drop the un-transformed numeric features
  select(-TOT_EMP_num, -JOBS_1000_num, -LOC_QUOTIENT_num)

# Verify updated dataset structure
glimpse(df_model_prep)

### 6.3 Occupation Groups

The first two digits of `OCC_CODE` are used to create `OCC_GROUP`. State and occupation group are converted to factors, and the original occupation code is removed from the final model dataset.


In [ ]:
# Extract 2-digit occupation group, drop original OCC_CODE, and convert to factors
df_model_prep <- df_model_prep %>%
  mutate(
    OCC_GROUP  = factor(substr(OCC_CODE, 1, 2)),
    PRIM_STATE = factor(PRIM_STATE)
  ) %>%
  select(-OCC_CODE)

# Inspect feature data types and summary
glimpse(df_model_prep)
summary(df_model_prep)

## 7. Train/Test Split

An **80/20 stratified split** is used so that the distribution of `High_Pay` is maintained between the training and test sets.


In [ ]:
library(rsample)

set.seed(123)
data_split <- initial_split(df_model_prep, prop = 0.80, strata = High_Pay)
train_data <- training(data_split)
test_data  <- testing(data_split)

cat("Training rows:", nrow(train_data), "| Testing rows:", nrow(test_data), "\n")

## 8. Baseline Logistic Regression

The baseline model predicts `High_Pay` using `log_TOT_EMP`, `log_JOBS_1000`, `log_LOC_QUOTIENT`, `PRIM_STATE`, and `OCC_GROUP`.


In [ ]:
# Fit Logistic Regression
logistic_model <- glm(
  High_Pay ~ log_TOT_EMP + log_JOBS_1000 + log_LOC_QUOTIENT + PRIM_STATE + OCC_GROUP,
  data = train_data,
  family = binomial(link = "logit")
)

# Model summary
summary(logistic_model)

## 9. Baseline Model Evaluation

The baseline model is evaluated using a confusion matrix, ROC-AUC, F1-score, and related classification metrics.


In [ ]:
library(pROC)
# 1. Predict probabilities on the test set
test_data$pred_prob <- predict(logistic_model, newdata = test_data, type = "response")

# 2. Classify using 0.5 probability threshold
test_data$pred_class <- factor(
  if_else(test_data$pred_prob >= 0.5, "High", "Not_High"),
  levels = c("Not_High", "High")
)

# 3. Print Confusion Matrix and Metrics
conf_matrix <- confusionMatrix(test_data$pred_class, test_data$High_Pay, positive = "High")
print(conf_matrix)

# 4. Calculate ROC-AUC Score
roc_obj <- roc(test_data$High_Pay, test_data$pred_prob, levels = c("Not_High", "High"))
cat("\nTest Set ROC-AUC Score:", round(auc(roc_obj), 4), "\n")

# 5. Plot ROC Curve
plot(roc_obj, main = "ROC Curve - Logistic Regression Model", col = "blue", lwd = 2)

In [ ]:
library(pROC)

# 1. Compute the ROC object using actual binary target and predicted probabilities
roc_obj <- roc(
  response = test_data$High_Pay, 
  predictor = test_data$pred_prob, 
  levels = c("Not_High", "High")
)

# 2. Extract and print the AUC score
auc_score <- auc(roc_obj)
cat("Test Set ROC-AUC Score:", round(auc_score, 4), "\n")

# 3. Plot the ROC Curve
plot(
  roc_obj, 
  main = paste0("ROC Curve (AUC = ", round(auc_score, 3), ")"), 
  col = "steelblue", 
  lwd = 3,
  legacy.axes = TRUE # Displays Specificity as 1 - Specificity (False Positive Rate)
)
abline(a = 0, b = 1, lty = 2, col = "red") # Diagonal baseline reference line

In [ ]:
# 1. Calculate F1-Score using caret
f1_val <- F_meas(data = test_data$pred_class, reference = test_data$High_Pay, relevant = "High")
cat("Test Set F1-Score:", round(f1_val, 4), "\n")

# 2. Alternatively, calculate manually from Precision and Recall
precision <- conf_matrix$byClass["Pos Pred Value"]
recall    <- conf_matrix$byClass["Sensitivity"]
f1_manual <- 2 * (precision * recall) / (precision + recall)
cat("Calculated F1-Score:", round(f1_manual, 4), "\n")

In [ ]:
# Extract coefficients, calculate odds ratios and 95% confidence intervals
odds_ratios <- data.frame(
  Feature = names(coef(logistic_model)),
  Estimate_LogOdds = coef(logistic_model),
  Odds_Ratio = exp(coef(logistic_model)),
  CI_Lower = exp(confint.default(logistic_model)[, 1]),
  CI_Upper = exp(confint.default(logistic_model)[, 2]),
  P_Value = summary(logistic_model)$coefficients[, 4]
) %>%
  mutate(across(c(Estimate_LogOdds, Odds_Ratio, CI_Lower, CI_Upper), ~ round(.x, 4))) %>%
  arrange(desc(Odds_Ratio))

# Display top factors increasing/decreasing odds of High_Pay
print(odds_ratios)

## 10. Model Interpretation: Odds Ratios

Odds ratios describe how the odds of being classified as high-paying change with the predictors, holding the other model variables constant.

For categorical predictors, odds ratios are interpreted relative to the model's reference category.


In [ ]:
# Compute GVIF for the logistic regression model
vif_results <- vif(logistic_model)
print(vif_results)

# Format into a clean dataframe with standard VIF adjusted for degrees of freedom
vif_df <- as.data.frame(vif_results)
if ("GVIF^(1/(2*Df))" %in% colnames(vif_df)) {
  vif_df$Adjusted_VIF <- vif_df$`GVIF^(1/(2*Df))`^2
}
print(vif_df)

## 11. Multicollinearity Diagnostics

Generalized variance inflation factors (GVIFs) are examined to identify problematic multicollinearity among predictors.


In [ ]:
# 1. Create interaction terms: Predictor * log(Predictor)
# Note: Adding a small constant if values are zero or near-zero
df_box_tidwell <- train_data %>%
  mutate(
    interaction_TOT_EMP   = log_TOT_EMP * log(log_TOT_EMP + 1e-5),
    interaction_JOBS_1000 = log_JOBS_1000 * log(log_JOBS_1000 + 1e-5),
    interaction_LOC_QUO   = log_LOC_QUOTIENT * log(log_LOC_QUOTIENT + 1e-5)
  )

# 2. Fit full model including original terms, interactions, and categorical controls
box_tidwell_model <- glm(
  High_Pay ~ log_TOT_EMP + log_JOBS_1000 + log_LOC_QUOTIENT + 
             interaction_TOT_EMP + interaction_JOBS_1000 + interaction_LOC_QUO + 
             PRIM_STATE + OCC_GROUP,
  data = df_box_tidwell,
  family = binomial(link = "logit")
)

# 3. Extract p-values for the interaction terms
bt_summary <- summary(box_tidwell_model)$coefficients
interaction_rows <- c("interaction_TOT_EMP", "interaction_JOBS_1000", "interaction_LOC_QUO")

print(bt_summary[rownames(bt_summary) %in% interaction_rows, ])

## 12. Logistic Regression Linearity Assumption

The Box-Tidwell-style diagnostic is used to assess whether the continuous predictors have an approximately linear relationship with the log-odds.

The original analysis found:

- `log_TOT_EMP`: no evidence of nonlinearity (`p = 0.8043`)
- `log_JOBS_1000`: strong evidence of nonlinearity (`p = 2.846e-13`)
- `log_LOC_QUOTIENT`: no evidence of nonlinearity (`p = 0.5374`)


In [ ]:
library(pROC)

# 1. Refit logistic regression model with non-linear polynomial term
logistic_model_poly <- glm(
  High_Pay ~ log_TOT_EMP + log_JOBS_1000 + I(log_JOBS_1000^2) + log_LOC_QUOTIENT + 
             PRIM_STATE + OCC_GROUP,
  data = train_data,
  family = binomial(link = "logit")
)

# 2. Inspect coefficients for linear and quadratic log_JOBS_1000 terms
poly_summary <- summary(logistic_model_poly)$coefficients
print(poly_summary[grep("log_JOBS_1000", rownames(poly_summary)), ])

# 3. Predict on the test set with the updated model
test_data$pred_prob_poly <- predict(logistic_model_poly, newdata = test_data, type = "response")
test_data$pred_class_poly <- factor(
  if_else(test_data$pred_prob_poly >= 0.5, "High", "Not_High"),
  levels = c("Not_High", "High")
)

# 4. Compare Performance Metrics
# Confusion Matrix & Accuracy
conf_matrix_poly <- confusionMatrix(test_data$pred_class_poly, test_data$High_Pay, positive = "High")
print(conf_matrix_poly$overall["Accuracy"])

# ROC-AUC
roc_poly <- roc(test_data$High_Pay, test_data$pred_prob_poly, levels = c("Not_High", "High"))
cat("Updated Test Set ROC-AUC:", round(auc(roc_poly), 4), "\n")

# Likelihood Ratio Test (compare baseline vs polynomial model)
anova(logistic_model, logistic_model_poly, test = "Chisq")

## 13. Final Logistic Regression Model

Because `log_JOBS_1000` showed evidence of nonlinearity, a quadratic term is added:

`I(log_JOBS_1000^2)`

The polynomial model is compared with the baseline model using a likelihood-ratio test.


In [ ]:
library(ggplot2)
library(dplyr)
library(knitr)

# ==============================================================================
# 1. VISUALIZE NON-LINEAR EFFECT CURVE FOR log_JOBS_1000
# ==============================================================================

# Create a sequence across the range of log_JOBS_1000 while holding other features constant
jobs_seq <- seq(
  min(train_data$log_JOBS_1000, na.rm = TRUE), 
  max(train_data$log_JOBS_1000, na.rm = TRUE), 
  length.out = 300
)

# Representative baseline profile (median values for numeric, modal categories for factors)
effect_data <- data.frame(
  log_JOBS_1000    = jobs_seq,
  log_TOT_EMP      = median(train_data$log_TOT_EMP, na.rm = TRUE),
  log_LOC_QUOTIENT = median(train_data$log_LOC_QUOTIENT, na.rm = TRUE),
  PRIM_STATE       = names(which.max(table(train_data$PRIM_STATE))),
  OCC_GROUP        = names(which.max(table(train_data$OCC_GROUP)))
)

# Predict log-odds and calculate 95% Confidence Intervals
pred_link <- predict(logistic_model_poly, newdata = effect_data, type = "link", se.fit = TRUE)

effect_data <- effect_data %>%
  mutate(
    fit_link    = pred_link$fit,
    se_link     = pred_link$se.fit,
    pred_prob   = plogis(fit_link),
    ci_lower    = plogis(fit_link - 1.96 * se_link),
    ci_upper    = plogis(fit_link + 1.96 * se_link)
  )

# Plot Partial Dependence Curve
ggplot(effect_data, aes(x = log_JOBS_1000, y = pred_prob)) +
  geom_ribbon(aes(ymin = ci_lower, ymax = ci_upper), fill = "steelblue", alpha = 0.2) +
  geom_line(color = "#1f4e78", size = 1.2) +
  labs(
    title = "Non-Linear Effect of Job Density (log_JOBS_1000) on High_Pay Probability",
    subtitle = "Partial Dependence Plot with 95% Confidence Bands",
    x = "log(JOBS_1000 + 1)",
    y = "Predicted Probability of High_Pay"
  ) +
  theme_minimal(base_size = 12)

# ==============================================================================
# 2. FINAL MODEL PERFORMANCE SUMMARY TABLE
# ==============================================================================

summary_metrics <- data.frame(
  Metric = c("Accuracy", "ROC-AUC", "F1-Score", "Sensitivity (Recall)", "Specificity", "Precision (PPV)", "Cohen's Kappa"),
  Value = c("78.02%", "0.8585", "0.7754", "75.83%", "80.21%", "79.31%", "0.5604"),
  Interpretation = c(
    "Overall correct classification rate on unseen test data.",
    "Excellent discrimination capability across probability thresholds.",
    "Strong balance between Precision and Recall.",
    "Proportion of actual High-Pay occupations correctly identified.",
    "Proportion of non-High-Pay occupations correctly identified.",
    "Probability that an occupation flagged as High-Pay is actually High-Pay.",
    "Moderate-to-substantial agreement beyond random chance."
  )
)

kable(summary_metrics, caption = "Final Logistic Regression Model Performance Summary")

## 14. Nonlinear Effect Visualization

In [ ]:
# ==============================================================================
# 1. GENERATE PREDICTIONS FOR FULL DATASET
# ==============================================================================

# Calculate predicted probability of High_Pay across all state-job combinations
df_predictions <- df_model_prep %>%
  mutate(
    High_Pay_Prob = predict(logistic_model_poly, newdata = ., type = "response"),
    Predicted_Class = if_else(High_Pay_Prob >= 0.5, "High_Pay", "Not_High_Pay")
  )

# ==============================================================================
# 2. STATE-LEVEL SUMMARY: HIGH-PAY OCCUPATION DENSITY
# ==============================================================================

# Aggregate predictions by state to find regions with highest high-pay concentrations
state_summary <- df_predictions %>%
  group_by(PRIM_STATE) %>%
  summarise(
    Total_Occupations    = n(),
    High_Pay_Count       = sum(Predicted_Class == "High_Pay"),
    Pct_High_Pay         = mean(Predicted_Class == "High_Pay") * 100,
    Avg_High_Pay_Prob    = mean(High_Pay_Prob)
  ) %>%
  arrange(desc(Pct_High_Pay))

cat("--- Top 10 States with Highest Share of High-Pay Occupations ---\n")
print(head(state_summary, 10))

# ==============================================================================
# 3. OCCUPATION GROUP SUMMARY: MOST LUCRATIVE JOB FAMILIES
# ==============================================================================

# Aggregate predictions by 2-digit occupation family (OCC_GROUP)
group_summary <- df_predictions %>%
  group_by(OCC_GROUP) %>%
  summarise(
    Total_Jobs           = n(),
    High_Pay_Count       = sum(Predicted_Class == "High_Pay"),
    Pct_High_Pay         = mean(Predicted_Class == "High_Pay") * 100,
    Avg_High_Pay_Prob    = mean(High_Pay_Prob)
  ) %>%
  arrange(desc(Avg_High_Pay_Prob))

cat("\n--- Top 10 Occupation Groups by High-Pay Likelihood ---\n")
print(head(group_summary, 10))

# ==============================================================================
# 4. IDENTIFY TOP HIGH-PAY OPPORTUNITIES BY STATE & JOB GROUP
# ==============================================================================

# Extract the top 5 highest-probability state-job combinations per state
top_jobs_per_state <- df_predictions %>%
  group_by(PRIM_STATE) %>%
  slice_max(order_by = High_Pay_Prob, n = 5) %>%
  select(PRIM_STATE, OCC_GROUP, log_TOT_EMP, log_LOC_QUOTIENT, High_Pay_Prob)

cat("\n--- Top High-Pay Job Groups in California (CA) ---\n")
print(filter(top_jobs_per_state, PRIM_STATE == "CA"))

## 15. Final Model Predictions

Predicted probabilities are generated for the full cleaned modeling dataset. A probability of **0.50 or greater** is classified as `High_Pay`.


In [ ]:
library(gt)

state_summary %>%
  head(10) %>%
  mutate(
    Pct_High_Pay = paste0(round(Pct_High_Pay, 1), "%"),
    Avg_High_Pay_Prob = round(Avg_High_Pay_Prob, 3)
  ) %>%
  gt() %>%
  tab_header(
    title = "Top 10 States by High-Pay Occupation Share"
  ) %>%
  cols_label(
    PRIM_STATE = "State",
    Total_Occupations = "Total Occupations",
    High_Pay_Count = "High-Pay Count",
    Pct_High_Pay = "High-Pay Share",
    Avg_High_Pay_Prob = "Avg. Probability"
  )

## 16. Key Results

### Final Model Performance

| Metric | Result |
|---|---:|
| Accuracy | **78.02%** |
| ROC-AUC | **0.8585** |
| F1-Score | **0.7754** |
| Sensitivity | **75.83%** |
| Specificity | **80.21%** |
| Precision / PPV | **79.31%** |
| Cohen's Kappa | **0.5604** |

The polynomial term for `log_JOBS_1000` significantly improved statistical fit:

- Deviance improvement: **74.00**
- Likelihood-ratio test p-value: **7.81 × 10⁻¹⁸**

However, the baseline model had a slightly higher test ROC-AUC (**0.8591**) than the polynomial model (**0.8585**). This shows that improved statistical fit does not necessarily translate into improved held-out classification discrimination.


## 17. Example Findings

The state-level predictions identify **DC, Washington, Alaska, Massachusetts, and California** among the states with the highest predicted proportions of high-paying occupations.

The occupation-group predictions show especially high predicted probabilities for groups such as:

- 11 — Management
- 15 — Computer and Mathematical
- 17 — Architecture and Engineering
- 13 — Business and Financial Operations
- 19 — Life, Physical, and Social Science

The California analysis provides an example of using model probabilities to identify occupation groups with particularly high predicted probabilities.


## 18. Limitations

- The target is defined relative to the median wage in this dataset rather than an external definition of a high-paying occupation.
- Logistic regression estimates associations and should not be interpreted as establishing causality.
- State and occupation-group effects depend on the reference categories used by the model.
- The polynomial term improved statistical fit but did not improve test ROC-AUC.
- The model does not include all potentially relevant economic, demographic, educational, industry, or occupational characteristics.
- The dataset represents a particular reporting period and may not generalize to other years without validation.


## 19. Conclusion

This project demonstrates an end-to-end **logistic regression classification workflow in R** for predicting high-paying occupations.

The workflow combines data cleaning, exploratory analysis, feature engineering, assumption checking, multicollinearity diagnostics, nonlinear modeling, classification evaluation, odds-ratio interpretation, and geographic/occupational prediction summaries.

The final model achieves a test **ROC-AUC of 0.8585**, indicating strong discrimination between the two wage classes in the held-out test data.
